# 🧠 Chain of Thought Prompting

**Exercise duration: ~2 minutes** | Run every cell top-to-bottom

---

## What You'll Discover

A single sentence added to a prompt can turn a confused, error-prone model into a careful, step-by-step problem solver. We'll test this on **GSM8K** math word problems.

---

**How to use this notebook:**
- **[RUN]** cells — just execute, no edits needed.
- **[TODO]** cells — fill in the missing piece before running.

> ⚠️ Use a **GPU runtime**: Runtime → Change runtime type → T4 GPU.

## 1. Setup

**[RUN]** Install packages, check the GPU, load the model.

In [ ]:
!pip install -q transformers accelerate datasets
import torch
print(f"\u2705 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(10)+chr(10)+'  WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU.'}")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # half-precision: 2x faster, half the VRAM
    device_map="auto",           # auto-selects GPU
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
import re
import uuid, time, html as html_lib
from IPython.display import display, HTML


def clean_model_text(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()

    # Unwrap common LaTeX answer wrappers.
    # Examples: "\\boxed{42}", "$\\boxed{42}$", "boxed{42}", "$42$", "\\(42\\)", "\\[42\\]".
    t = re.sub(r"\$?\\\\boxed\{([^}]*)\}\$?", r"\1", t)
    t = re.sub(r"\$?boxed\{([^}]*)\}\$?", r"\1", t)

    m = re.fullmatch(r"\$([^$]+)\$", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\((.*)\\\)", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\[(.*)\\\]", t)
    if m:
        t = m.group(1).strip()

    return t


def generate_response(messages, max_new_tokens=None, creative=False):
    """Send chat-formatted messages to the model and return the response."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = dict(
        do_sample=creative,
        temperature=0.7 if creative else 1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    if max_new_tokens is not None:
        gen_kwargs["max_new_tokens"] = max_new_tokens

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return clean_model_text(response)


def display_sample(question, answer):
    display(HTML(
        f"<p><strong>Question:</strong> {question}</p>"
        f"<p><strong>Answer:</strong> {answer}</p>"
    ))


def display_response(prompt_text, response_text, elapsed=None):
    uid = str(uuid.uuid4()).replace("-", "")
    time_tag = f"<div style='color:#666;font-size:90%;margin-top:4px'>&#x23F1; {elapsed:.1f}s</div>" if elapsed else ""
    safe_p = html_lib.escape(str(prompt_text).strip())
    safe_r = html_lib.escape(clean_model_text(str(response_text))).strip()
    display(HTML(
        "<style>.rt{border-collapse:collapse;width:100%;margin:10px 0;font-family:'Segoe UI',sans-serif;font-size:14px}"
        ".rt th,.rt td{border:1px solid #ddd;padding:9px 12px;vertical-align:top}"
        ".rt th{background:#f5f5f5;width:120px;font-weight:600}"
        "pre.rm{white-space:pre-wrap;margin:0}</style>"
        f"<table class='rt'>"
        f"<tr><th>Prompt</th><td><pre class='rm'>{safe_p}</pre></td></tr>"
        f"<tr><th>Response</th><td><pre class='rm'>{safe_r}</pre>{time_tag}</td></tr>"
        "</table>"
    ))


def run_and_display(messages, max_new_tokens=None, creative=False):
    """Generate and display in a table. Returns the response string."""
    prompt_text = messages[-1]["content"]
    t0 = time.time()
    response = generate_response(messages, max_new_tokens=max_new_tokens, creative=creative)
    elapsed = time.time() - t0
    display_response(prompt_text, response, elapsed)
    return response

## 2. Chain of Thought Prompting

### The big idea

When you ask someone a hard question and just say *"Answer this"*, they might guess. But say *"Think through it step by step"* and they slow down, reason carefully, and are far more accurate.

**Chain of Thought (CoT) prompting does exactly this for LLMs.** A short trigger phrase nudges the model to reason explicitly before committing to an answer.

Let's see it in action.

**[RUN]** Load the GSM8K dataset and peek at a sample question.

In [ ]:
ds_main = load_dataset("openai/gsm8k", "main")
train_split_main = ds_main["train"]
test_split_main  = ds_main["test"]
print(f"Train: {len(train_split_main)} | Test: {len(test_split_main)}")

In [ ]:
display_sample(train_split_main[0]["question"], train_split_main[0]["answer"])

**[RUN]** Ask the model directly — zero-shot, no guidance.

In [ ]:
question = train_split_main[0]["question"]
messages = [{"role": "user", "content": question}]
run_and_display(messages, max_new_tokens=200)

Not great 😕 — the model jumped to an answer without showing its reasoning. This is *pattern matching without thinking*.

### ✏️ [TODO] Add a CoT trigger

Complete `cot_prompt` with a short phrase that encourages the model to reason step by step **before** answering.

> 💡 **Hint:** Think how you'd tell a student to slow down and show their work. Even 4-5 words work!

In [ ]:
cot_prompt = ""  # TODO: Add a Chain of Thought trigger phrase here

messages = [{"role": "user", "content": f"{cot_prompt}\n{question}"}]
run_and_display(messages, max_new_tokens=200)

🎉 Much better! The model now reasons step by step before answering.

This is Chain of Thought prompting: **words shape thinking**, even for AI. The model's weights didn't change — only the prompt did.